In [35]:
import pandas as pd
import numpy as np
import unicodedata
from difflib import get_close_matches
import re
import string

## Merge avec gestion des doublons

In [ ]:
# Charger les deux fichiers CSV
df1 = pd.read_csv('data/processed/offres_it_ciblees_france_travail.csv')
df2 = pd.read_csv('data/processed/offres_it_france_linkedin.csv')

print("AVANT MERGE:")
print(f"Fichier 1: {df1.shape[0]} lignes, {df1.shape[1]} colonnes")
print(f"Fichier 2: {df2.shape[0]} lignes, {df2.shape[1]} colonnes")

# Vérification que les colonnes sont compatibles
cols1 = set(df1.columns)
cols2 = set(df2.columns)

print(f"\nVÉRIFICATION:")
print(f"Colonnes communes: {len(cols1 & cols2)}")
print(f"Colonnes uniquement dans df1: {cols1 - cols2}")
print(f"Colonnes uniquement dans df2: {cols2 - cols1}")

# Merge simple (concaténation verticale)
df_merged = pd.concat([df1, df2], ignore_index=True)

print(f"\nAPRÈS MERGE:")
print(f"Taille totale: {df_merged.shape[0]} lignes, {df_merged.shape[1]} colonnes")

# Vérification des doublons
doublons = df_merged.duplicated(subset=['id_offre']).sum()
print(f"Doublons sur id_offre: {doublons}")

# Aperçu du résultat
print("\nAPERÇU DU RÉSULTAT:")
print(df_merged.head(3))

# Optionnel: Sauvegarder le résultat
df_merged.to_csv('data_fusionne.csv', index=False)
print(f"\n Fichier fusionné sauvegardé: 'data_fusionne.csv'")

AVANT MERGE:
Fichier 1: 5077 lignes, 14 colonnes
Fichier 2: 319 lignes, 14 colonnes

VÉRIFICATION:
Colonnes communes: 14
Colonnes uniquement dans df1: set()
Colonnes uniquement dans df2: set()

APRÈS MERGE:
Taille totale: 5396 lignes, 14 colonnes
Doublons sur id_offre: 1484

APERÇU DU RÉSULTAT:
                                      intitule_poste  nom_entreprise  \
0  Développeur C embarqué - Domaine des systèmes ...         SILKHOM   
1     Developpeur Fullstack Java / Angular H/F (H/F)  STEP UP NANTES   
2        Développeur / Développeuse full-stack (H/F)  TECH IN FRANCE   

         ville_region          date_publication type_contrat  \
0          91 - Massy  2025-11-25T18:04:23.304Z          CDI   
1         44 - NANTES  2025-11-25T15:37:12.545Z          CDI   
2  75 - Paris (Dept.)  2025-11-25T15:06:02.674Z          CDI   

  experience_demandee niveau_seniorite  \
0    Débutant accepté           Junior   
1             4 An(s)         Confirmé   
2             5 An(s)     Non sp

## Colonne teletravail

In [4]:
# Récupérer toutes les valeurs distinctes de la colonne teletravail
valeurs_teletravail = df_merged['teletravail'].unique()

print(" VALEURS DISTINCTES DANS LA COLONNE 'TÉLÉTRAVAIL':")
print("=" * 50)
print(valeurs_teletravail)

# Version avec plus de détails
print("\n ANALYSE DÉTAILLÉE DE LA COLONNE 'TÉLÉTRAVAIL':")
print("=" * 50)

# Compter le nombre d'occurrences de chaque valeur
comptage = df_merged['teletravail'].value_counts()
print("Répartition des valeurs:")
print(comptage)

# Pourcentage de chaque valeur
pourcentages = df_merged['teletravail'].value_counts(normalize=True) * 100
print("\nPourcentages:")
print(pourcentages.round(2).astype(str) + " %")

# Information sur les valeurs manquantes
valeurs_manquantes = df_merged['teletravail'].isna().sum()
print(f"\n Valeurs manquantes: {valeurs_manquantes}")
print(f" Total des lignes analysées: {len(df_merged)}")

 VALEURS DISTINCTES DANS LA COLONNE 'TÉLÉTRAVAIL':
['Non' 'Oui']

 ANALYSE DÉTAILLÉE DE LA COLONNE 'TÉLÉTRAVAIL':
Répartition des valeurs:
teletravail
Non    3931
Oui    1465
Name: count, dtype: int64

Pourcentages:
teletravail
Non    72.85 %
Oui    27.15 %
Name: proportion, dtype: object

 Valeurs manquantes: 0
 Total des lignes analysées: 5396


## Colonne Niveau Seniorite

In [5]:
valeurs_seniorite = df_merged['niveau_seniorite'].unique()

print("VALEURS DISTINCTES - NIVEAU SÉNIORITÉ:")
print("=" * 50)
print(valeurs_seniorite)

VALEURS DISTINCTES - NIVEAU SÉNIORITÉ:
['Junior' 'Confirmé' 'Non spécifié' 'Senior']


## Colonne Type de contrat

In [6]:
# Méthode simple avec value_counts()
valeurs_avec_occurrences = df_merged['type_contrat'].value_counts()

print("VALEURS DISTINCTES AVEC OCCURRENCES - TYPE CONTRAT:")
print("=" * 60)

for valeur, occurrence in valeurs_avec_occurrences.items():
    pourcentage = (occurrence / len(df_merged)) * 100
    print(f"• '{valeur}' : {occurrence} occurrences ({pourcentage:.1f}%)")

VALEURS DISTINCTES AVEC OCCURRENCES - TYPE CONTRAT:
• 'CDI' : 4408 occurrences (81.7%)
• 'CDD - 12 Mois' : 245 occurrences (4.5%)
• 'CDD - 36 Mois' : 203 occurrences (3.8%)
• 'Profession libérale' : 100 occurrences (1.9%)
• 'CDD - 6 Mois' : 82 occurrences (1.5%)
• 'Intérim - 12 Mois' : 47 occurrences (0.9%)
• 'Intérim - 6 Mois' : 46 occurrences (0.9%)
• 'Intérim - 3 Mois' : 37 occurrences (0.7%)
• 'Stage' : 27 occurrences (0.5%)
• 'Intérim - 4 Mois' : 17 occurrences (0.3%)
• 'Intérim - 18 Mois' : 16 occurrences (0.3%)
• 'CDD - 4 Mois' : 13 occurrences (0.2%)
• 'CDD - 8 Mois' : 12 occurrences (0.2%)
• 'CDD - 3 Mois' : 12 occurrences (0.2%)
• 'Intérim - 1 Mois' : 12 occurrences (0.2%)
• 'CDD - 24 Mois' : 11 occurrences (0.2%)
• 'Contrat à durée indéterminée' : 11 occurrences (0.2%)
• 'Intérim - 2 Mois' : 9 occurrences (0.2%)
• 'CDD - 10 Mois' : 7 occurrences (0.1%)
• 'CDD - 9 Mois' : 7 occurrences (0.1%)
• 'CDD - 1 Mois' : 6 occurrences (0.1%)
• 'CDD - 2 Mois' : 5 occurrences (0.1%)
• 'C

In [7]:
# Fonction de normalisation directe
def normaliser_type_contrat(valeur):
    """
    Normalise une valeur de type_contrat vers les 6 catégories cibles
    """
    if pd.isna(valeur):
        return 'Non spécifié'

    valeur_str = str(valeur).lower().strip()

    # Détection par patterns
    if re.search(r'\bcdi\b|\bcontrat à durée indéterminée\b', valeur_str):
        return 'CDI'
    elif re.search(r'\bcdd\b|\bcontrat à durée déterminée\b', valeur_str):
        return 'CDD'
    elif re.search(r'\bintérim\b|\binterim\b|\btemporaire\b', valeur_str):
        return 'Intérim'
    elif re.search(r'\bstage\b|\binternship\b|\bstagiaire\b', valeur_str):
        return 'Stage'
    elif re.search(r'\blibérale\b', valeur_str):
        return 'Profession libérale'
    elif re.search(r'\bcommerciale\b', valeur_str):
        return 'Profession commerciale'
    else:
        return 'Autre'


# APPLICATION DIRECTE À LA COLONNE EXISTANTE
print("NORMALISATION DE LA COLONNE TYPE_CONTRAT")
print("=" * 50)

# on remplace directement la colonne existante au lieu d'en créer une nouvelle
df_merged['type_contrat'] = df_merged['type_contrat'].apply(normaliser_type_contrat)

# Afficher le résultat
stats_normalise = df_merged['type_contrat'].value_counts()
total = len(df_merged)

print("RÉSULTATS APRÈS NORMALISATION:")
print("-" * 40)

for categorie in ['CDI', 'CDD', 'Intérim', 'Stage', 'Profession libérale', 'Profession commerciale', 'Autre', 'Non spécifié']:
    if categorie in stats_normalise:
        count = stats_normalise[categorie]
        pourcentage = (count / total) * 100
        print(f"• {categorie:25} : {count:4d} occurrences ({pourcentage:5.1f}%)")

# Vérification rapide
print(f"\n VÉRIFICATION:")
print(f"Total des offres: {total}")
print(f"Catégories obtenues: {len(stats_normalise)}")


NORMALISATION DE LA COLONNE TYPE_CONTRAT
RÉSULTATS APRÈS NORMALISATION:
----------------------------------------
• CDI                       : 4420 occurrences ( 81.9%)
• CDD                       :  633 occurrences ( 11.7%)
• Intérim                   :  211 occurrences (  3.9%)
• Stage                     :   27 occurrences (  0.5%)
• Profession libérale       :  100 occurrences (  1.9%)
• Profession commerciale    :    5 occurrences (  0.1%)

 VÉRIFICATION:
Total des offres: 5396
Catégories obtenues: 6


## Colonne metier recherché

In [8]:
# Récupérer les valeurs distinctes AVEC leurs occurrences
metier_recherche_occurrences = df_merged['metier_recherche'].value_counts()

print(" VALEURS DISTINCTES AVEC OCCURRENCES - METIER RECHERCHE:")
print("=" * 70)

for valeur, occurrence in metier_recherche_occurrences.items():
    pourcentage = (occurrence / len(df_merged)) * 100
    print(f"• '{valeur}' : {occurrence} occurrences ({pourcentage:.1f}%)")

 VALEURS DISTINCTES AVEC OCCURRENCES - METIER RECHERCHE:
• 'développeur' : 326 occurrences (6.0%)
• 'data analyst' : 184 occurrences (3.4%)
• 'devops' : 181 occurrences (3.4%)
• 'cybersécurité' : 179 occurrences (3.3%)
• 'data scientist' : 163 occurrences (3.0%)
• 'ingénieur informatique' : 150 occurrences (2.8%)
• 'ingénieur logiciel' : 150 occurrences (2.8%)
• 'développeur web' : 150 occurrences (2.8%)
• 'développeur fullstack' : 150 occurrences (2.8%)
• 'ingénieur devops' : 150 occurrences (2.8%)
• 'spécialiste cloud' : 150 occurrences (2.8%)
• 'cloud engineer' : 150 occurrences (2.8%)
• 'ingénieur cloud' : 150 occurrences (2.8%)
• 'administrateur système' : 150 occurrences (2.8%)
• 'administrateur réseau' : 150 occurrences (2.8%)
• 'intelligence artificielle' : 150 occurrences (2.8%)
• 'webmaster' : 150 occurrences (2.8%)
• 'technicien informatique' : 150 occurrences (2.8%)
• 'support technique' : 150 occurrences (2.8%)
• 'ingénieur systèmes' : 150 occurrences (2.8%)
• 'développeur

In [9]:
def regrouper_vers_9_metiers(df):
    """
    Regroupe tous les métiers vers 9 catégories principales
    """
    df_regroupe = df.copy()

    # Mapping vers les 9 catégories principales
    def categoriser_metier(metier):
        metier_str = str(metier).lower().strip()

        # 1. Développement (regroupe tous les développeurs)
        if any(mot in metier_str for mot in ['développeur', 'dev', 'fullstack', 'frontend', 'backend', 'web', 'mobile', 'java', 'php', 'javascript', 'python', 'c#', 'intégrateur']):
            return 'Développement'

        # 2. Data (regroupe tous les métiers data)
        elif any(mot in metier_str for mot in ['data', 'analyste', 'scientifique', 'big data', 'data analyst', 'data scientist', 'data engineer', 'architecte data']):
            return 'Data'

        # 3. Cloud & DevOps
        elif any(mot in metier_str for mot in ['cloud', 'devops']):
            return 'Cloud & DevOps'

        # 4. Cybersécurité
        elif any(mot in metier_str for mot in ['cybersécurité', 'sécurité', 'pentester']):
            return 'Cybersécurité'

        # 5. Infrastructure & Support
        elif any(mot in metier_str for mot in ['administrateur', 'technicien', 'support', 'helpdesk', 'système', 'réseau', 'dba', 'webmaster']):
            return 'Infrastructure & Support'

        # 6. Ingénierie
        elif any(mot in metier_str for mot in ['ingénieur']):
            return 'Ingénierie'

        # 7. Intelligence Artificielle
        elif any(mot in metier_str for mot in ['intelligence artificielle', 'deep learning']):
            return 'Intelligence Artificielle'

        # 8. Design
        elif any(mot in metier_str for mot in ['designer', 'ux/ui']):
            return 'Design'

        # 9. Autres
        else:
            return 'Autres'

    # **Affectation directe à la colonne 'metier_recherche'**
    df_regroupe['metier_recherche'] = df_regroupe['metier_recherche'].apply(categoriser_metier)

    return df_regroupe

# Appliquer le regroupement
df_merged = regrouper_vers_9_metiers(df_merged)

# Vérifier le résultat
categories_finales = df_merged['metier_recherche'].value_counts()

print("LES 9 CATÉGORIES DE MÉTIERS FINALES:")
print("=" * 50)

total_offres = len(df_merged)
for categorie, occurrence in categories_finales.items():
    pourcentage = (occurrence / total_offres) * 100
    print(f"• {categorie:25} : {occurrence} occurrences ({pourcentage:.1f}%)")


LES 9 CATÉGORIES DE MÉTIERS FINALES:
• Développement             : 1954 occurrences (36.2%)
• Infrastructure & Support  : 987 occurrences (18.3%)
• Cloud & DevOps            : 750 occurrences (13.9%)
• Data                      : 741 occurrences (13.7%)
• Cybersécurité             : 483 occurrences (9.0%)
• Ingénierie                : 300 occurrences (5.6%)
• Intelligence Artificielle : 152 occurrences (2.8%)
• Design                    : 29 occurrences (0.5%)


## Colonne Expérience demandée

In [10]:
valeurs_experience = df_merged['experience_demandee'].unique()

print("VALEURS DISTINCTES - EXPERIENCE DEMANDEE:")
print("=" * 50)
print(valeurs_experience)

VALEURS DISTINCTES - EXPERIENCE DEMANDEE:
['Débutant accepté' '4 An(s)' '5 An(s)' '8 An(s)' '36 Mois' '3 An(s)'
 '7 An(s)' '2 An(s)' 'Débutant accepté - Réservé aux bénéficiaires RQTH '
 '1 An(s)' '24 Mois' "2 An(s) - Idéalement en agence d'intérim " '3 Mois'
 '48 Mois' '1 An(s) - sur même type de poste' '1 An(s) - commercial BtoB'
 '3 An(s) - sur LARAVEL et 1 an sur VUE.JS' '1 Mois'
 'Débutant accepté - Première expérience est un plus.' '12 Mois' '6 An(s)'
 '7 An(s) - développement TypeScript fullstack'
 "6 Mois - Sécurité électronique, de l'IoT " 'Expérience souhaitée'
 'Expérience exigée' nan '3 An(s) - réalisations concrètes à présenter'
 '4 An(s) - Windev' '3 An(s) - maitrise Langage C#' '6 Mois' '10 An(s)'
 '5 An(s) - SUR MEME FONCTION'
 'Débutant accepté - connaissance IA et Logic.Obligatoire'
 '2 An(s) - chef de projet ou assistant ' '10 Mois'
 "1 An(s) - De 1 à 3 ans d'expérience"
 '3 An(s) - Ingénierie IT ou dev logiciel'
 '3 An(s) - sur même type de poste' '4 An(s) - sur pos

In [11]:
def normaliser_experience_demandee(df):
    """
    Normalise la colonne experience_demandee et crée min/max experience
    """
    import re
    import pandas as pd

    def extraire_intervalle(valeur):
        """
        Extrait l'intervalle min/max d'expérience d'une valeur texte
        """
        if pd.isna(valeur) or str(valeur).strip().lower() == 'non spécifié':
            return 0, 2, 'Non spécifié'

        valeur_str = str(valeur).lower().strip()

        # 1. CAS DÉBUTANT
        if 'débutant' in valeur_str:
            return 0, 2, 'Débutant (0-2 ans)'

        # 2. CAS EXPÉRIENCE EXIGÉE/SOUHAITÉE
        if 'expérience exigée' in valeur_str:
            return 2, 5, 'Expérience exigée (2-5 ans)'
        if 'expérience souhaitée' in valeur_str:
            return 1, 3, 'Expérience souhaitée (1-3 ans)'

        # 3. CAS PLAGES EXPLICITES "X-Y ans"
        match_plage = re.search(r'(\d+)\s*-\s*(\d+)\s*ans?', valeur_str)
        if match_plage:
            min_exp = int(match_plage.group(1))
            max_exp = int(match_plage.group(2))
            return min_exp, max_exp, f'Plage ({min_exp}-{max_exp} ans)'

        # 4. CAS VALEURS AVEC "+" → "X+ ans"
        match_plus = re.search(r'(\d+)\s*\+\s*ans?', valeur_str)
        if match_plus:
            min_exp = int(match_plus.group(1))
            return min_exp, min_exp + 5, f'Minimum ({min_exp}+ ans)'

        # 5. EXTRACTION DES ANNÉES "X An(s)"
        match_ans = re.search(r'(\d+)\s*an\(s\)', valeur_str)
        if match_ans:
            annees = int(match_ans.group(1))
            return annees, annees, f'Précis ({annees} ans)'

        # 6. EXTRACTION DES MOIS "X Mois" → conversion en années
        match_mois = re.search(r'(\d+)\s*mois', valeur_str)
        if match_mois:
            mois = int(match_mois.group(1))
            annees = mois / 12
            if mois < 12:
                return round(annees, 1), round(annees + 0.5, 1), f'{mois} mois'
            else:
                annees_entier = mois // 12
                return annees_entier, annees_entier, f'{mois} mois ({annees_entier} ans)'

        # 7. EXTRACTION DE TOUT NOMBRE (dernier recours)
        nombres = re.findall(r'\d+', valeur_str)
        if nombres:
            nombre = int(nombres[0])
            if nombre < 10:
                return nombre, nombre, f'Estimé ({nombre} ans)'

        # 8. CAS PAR DÉFAUT
        return 0, 2, 'Non classé'

    # Appliquer la fonction directement sur df
    resultats = df['experience_demandee'].apply(extraire_intervalle)

    # Affectation des colonnes directement
    df['min_experience'] = [r[0] for r in resultats]
    df['max_experience'] = [r[1] for r in resultats]
    df['experience_normalisee'] = [r[2] for r in resultats]

    return df

# Appliquer la normalisation directement sur df_merged
df_merged = normaliser_experience_demandee(df_merged)

# Vérification des résultats
print("✅ RÉSULTATS APRÈS NORMALISATION:")
print("-" * 50)
print(df_merged[['experience_demandee', 'min_experience', 'max_experience', 'experience_normalisee']].head())


✅ RÉSULTATS APRÈS NORMALISATION:
--------------------------------------------------
  experience_demandee  min_experience  max_experience experience_normalisee
0    Débutant accepté             0.0             2.0    Débutant (0-2 ans)
1             4 An(s)             4.0             4.0        Précis (4 ans)
2             5 An(s)             5.0             5.0        Précis (5 ans)
3             8 An(s)             8.0             8.0        Précis (8 ans)
4             36 Mois             3.0             3.0       36 mois (3 ans)


In [13]:
df_merged.drop(columns=['experience_demandee', 'experience_normalisee'], inplace=True)


## Colonne nom d'entreprise

In [14]:
nb_nulls = df_merged['nom_entreprise'].isnull().sum()

print("Nombre de valeurs nulles dans nom_entreprise :", nb_nulls)


Nombre de valeurs nulles dans nom_entreprise : 1489


In [15]:
df_merged['nom_entreprise'] = df_merged['nom_entreprise'].fillna("INCONNU")


## Colonne Date Publication

In [19]:
def detecter_formats_dates(serie):
    """
    Analyse toutes les dates d'une colonne et retourne les différents formats distincts.
    """
    formats_detectes = {}

    for date in serie.astype(str):

        # Analyse de la longueur
        longueur = len(date)

        # Détecter présence de timezone
        has_z = date.endswith("Z")

        # Détecter présence de fraction de seconde
        frac_match = re.search(r'\.(\d+)', date)
        frac_len = len(frac_match.group(1)) if frac_match else 0

        # Création d'un identifiant unique du format
        key = (
            f"longueur={longueur}, "
            f"millisecondes={frac_len}, "
            f"Z={'oui' if has_z else 'non'}"
        )

        # Enregistrer un exemple pour ce format
        if key not in formats_detectes:
            formats_detectes[key] = date

    return formats_detectes

formats = detecter_formats_dates(df_merged['date_publication'])

print("FORMATS DE DATES DÉTECTÉS :")
for fmt, exemple in formats.items():
    print(f"- {fmt}  →  exemple : {exemple}")


FORMATS DE DATES DÉTECTÉS :
- longueur=24, millisecondes=3, Z=oui  →  exemple : 2025-11-25T18:04:23.304Z
- longueur=26, millisecondes=6, Z=non  →  exemple : 2025-11-26T11:53:31.844001


In [ ]:
def normaliser_dates(colonne):
    # Convertir toutes les dates en datetime (pandas gère les deux formats)
    dt = pd.to_datetime(colonne, errors='coerce')

    # Formater toutes les dates en format ISO avec microsecondes
    formatted = dt.dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')

    # Réduire à 3 millisecondes (de .ffffffZ à .fffZ)
    formatted = formatted.str.replace(
        r'(\.\d{3})\d{3}Z$',
        r'\1Z',
        regex=True
    )

    return formatted

# Appliquer sur df_merged
df_merged['date_publication'] = normaliser_dates(df_merged['date_publication'])


## Colonne Ville-Région

In [23]:
def remove_accents(s):
    if pd.isna(s):
        return s
    return ''.join(c for c in unicodedata.normalize('NFKD', s)
                   if not unicodedata.combining(c))

def nettoyer_nom_ville(v):
    s = str(v).strip()

    # Enlever "75 - " ou "974 - "
    s = re.sub(r'^\s*\d{2,3}\s*-\s*', '', s)

    # Enlever "périphérie", "agglo", "métropole", "arrondissement"…
    s = re.sub(r'(périphérie|peripherie|agglo|agglomération|metropole|métropole|environs)', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\b\d+(er|ème)? arrondissement\b', '', s, flags=re.IGNORECASE)

    # Enlever ponctuation
    s = re.sub(r"[\/,;-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return remove_accents(s).lower()

def extraire_departement(v):
    m = re.match(r'^\s*(\d{2,3})\s*-\s*', str(v))
    return m.group(1) if m else None

def generer_ville_et_departement(df):
    # 1) extraire département si présent
    df['departement'] = df['ville_region'].apply(extraire_departement)

    # 2) nettoyer nom de ville
    df['ville'] = df['ville_region'].apply(nettoyer_nom_ville)

    # 3) construire mapping ville -> département pour propagation
    mapping = (
        df.dropna(subset=['departement'])
          .groupby('ville')['departement']
          .agg(lambda x: x.value_counts().index[0])
          .to_dict()
    )

    # 4) remplir départements manquants
    def infer_dept(ville):
        if ville in mapping:
            return mapping[ville]
        match = get_close_matches(ville, mapping.keys(), n=1, cutoff=0.80)
        if match:
            return mapping[match[0]]
        return None

    df.loc[df['departement'].isna(), 'departement'] = \
        df.loc[df['departement'].isna(), 'ville'].apply(infer_dept)

    return df

# --- Application ---
df_merged = generer_ville_et_departement(df_merged)

In [40]:
df_merged = df_merged.drop('ville_region', axis=1)

## Colonne Description de Poste

In [30]:
def clean_description(text):
    text = str(text)
    # Supprimer les emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)

    # Supprimer les mots "Show moreShow less"
    text = re.sub(r"Show moreShow less", " ", text, flags=re.IGNORECASE)

    # Supprimer les sauts de ligne et tabulations
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # Supprimer les caractères spéciaux
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

    # Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text)

    # Mettre en minuscules
    text = text.lower().strip()

    return text

# Appliquer le nettoyage
df_merged['description_poste'] = df_merged['description_poste'].apply(clean_description)

## Colonne fourchette salariale

In [ ]:
col = "fourchette_salaire"   # 👉 remplace par ton nom de colonne

# Valeurs uniques avec occurrences
valeurs_avec_occ = df_merged[col].value_counts(dropna=False)

print(" Valeurs uniques avec occurrences :\n")
print(valeurs_avec_occ)

# Nombre de valeurs nulles
nb_null = df_merged[col].isna().sum()

print("\n Nombre de valeurs nulles :", nb_null)


📌 Valeurs uniques avec occurrences :

fourchette_salaire
NaN                                                      2725
Annuel de 42000 Euros à 55000 Euros sur 12.0 mois          57
Annuel de 48299 Euros à 63249 Euros sur 12.0 mois          53
Annuel de 40000.0 Euros à 50000.0 Euros sur 12.0 mois      44
Annuel de 45000.0 Euros à 65000.0 Euros sur 12.0 mois      40
                                                         ... 
Mensuel de 2300.0 Euros à 2600.0 Euros sur 12.0 mois        1
Horaire de 11.88 Euros à 12.0 Euros sur 12.0 mois           1
Annuel de 30000.0 Euros à 40000.0 Euros sur 13.3 mois       1
Horaire de 18.0 Euros à 20.33 Euros sur 12.0 mois           1
Annuel de 44 Euros à 49 Euros sur 12.0 mois                 1
Name: count, Length: 792, dtype: int64

📌 Nombre de valeurs nulles : 2725


In [36]:
# Parser complet en une fonction
def parser_salaire(s):
    if pd.isna(s): return np.nan, np.nan
    s = str(s).replace(',', '.')
    match = re.search(r'(Annuel|Mensuel|Horaire) de (\d+\.?\d*) Euros à (\d+\.?\d*) Euros sur (\d+\.?\d*) mois', s)
    if match:
        freq, low, high, months = match.groups()
        low, high, months = float(low), float(high), float(months)
        if freq == 'Mensuel': return low * months, high * months
        elif freq == 'Horaire': return low * 35 * 4.33 * months, high * 35 * 4.33 * months
        else: return low, high
    return np.nan, np.nan

# Application
salaires = df_merged['fourchette_salaire'].apply(parser_salaire)
df_merged['min_salaire'] = [s[0] for s in salaires]
df_merged['max_salaire'] = [s[1] for s in salaires]

# Valeurs par défaut
salaire_default = {'Développement': (35000,55000), 'Ingénierie': (40000,65000), 'Data': (38000,60000),
                   'Intelligence Artificielle': (45000,70000), 'Infrastructure & Support': (32000,48000),
                   'Cloud & DevOps': (42000,65000), 'Cybersécurité': (40000,62000), 'Design': (30000,45000)}

for metier, salaire in salaire_default.items():
    mask = df_merged['min_salaire'].isna() & (df_merged['metier_recherche'] == metier)
    df_merged.loc[mask, ['min_salaire', 'max_salaire']] = [salaire[0], salaire[1]]

# Finalisation
df_merged['min_salaire'] = df_merged['min_salaire'].fillna(35000)
df_merged['max_salaire'] = df_merged['max_salaire'].fillna(50000)
df_merged = df_merged.drop('fourchette_salaire', axis=1)

print("Salaires transformés avec succès!")

✅ Salaires transformés avec succès!


## Colonne Competences mentionnées

In [45]:
# Extraire toutes les compétences et les diviser
all_skills = df_merged['competences_mentionnees'].str.split(', ')

# Aplatir la liste et nettoyer les compétences
skills_flat = []
for skill_list in all_skills:
    if isinstance(skill_list, list):
        skills_flat.extend([skill.strip().lower() for skill in skill_list if skill.strip()])

# Convertir en set pour obtenir les compétences uniques
unique_skills = set(skills_flat)

# Afficher les résultats
nombre_competences_distinctes = len(unique_skills)
print(f"Nombre de compétences distinctes : {nombre_competences_distinctes}")
print("=" * 50)
print("Liste des compétences triées :")
print(sorted(unique_skills))

Nombre de compétences distinctes : 125
Liste des compétences triées :
['agile', 'ai', 'android', 'angular', 'ansible', 'ar/vr', 'artificial intelligence', 'asp.net', 'aws', 'azure', 'bash', 'bdd', 'big data', 'blockchain', 'bootstrap', 'c', 'c#', 'c++', 'cassandra', 'ci/cd', 'circleci', 'computer vision', 'confluence', 'cryptography', 'css', 'cybersecurity', 'cybersécurité', 'dart', 'data science', 'deep learning', 'devops', 'django', 'docker', 'dynamodb', 'elasticsearch', 'express', 'fastapi', 'firebase', 'firewall', 'flask', 'flutter', 'gcp', 'git', 'github actions', 'gitlab', 'go', 'grafana', 'graphql', 'hadoop', 'html', 'ionic', 'ios', 'iot', 'java', 'javascript', 'jenkins', 'jira', 'jquery', 'kafka', 'kanban', 'keras', 'kibana', 'kotlin', 'kubernetes', 'lambda', 'laravel', 'less', 'linux', 'machine learning', 'macos', 'matlab', 'microservices', 'mongodb', 'mysql', 'natural language processing', 'nestjs', 'next.js', 'nlp', 'node.js', 'oracle', 'owasp', 'perl', 'php', 'postgresql', 

In [51]:
import pandas as pd
import numpy as np

def create_final_dataframe(df, keep_original=True):
    """
    Crée le dataframe final avec les compétences en colonnes binaires
    """
    # Copie du dataframe
    if keep_original:
        df_final = df.copy()
    else:
        df_final = df.drop('competences_mentionnees', axis=1).copy()

    # Nettoyage des compétences
    skills_series = df['competences_mentionnees'].fillna('').str.split(',')

    # Création de la matrice one-hot
    all_skills = []
    for idx, skills in skills_series.items():
        skill_dict = {}
        if isinstance(skills, list):
            for skill in skills:
                cleaned_skill = skill.strip().lower()
                if cleaned_skill:
                    skill_dict[cleaned_skill] = 1
        all_skills.append(skill_dict)

    # Conversion en DataFrame
    skills_df = pd.DataFrame(all_skills).fillna(0).astype(int)

    # Concaténation
    final_df = pd.concat([df_final, skills_df], axis=1)

    return final_df

# Application
df_final = create_final_dataframe(df_merged, keep_original=False)

print(" Transformation réussie !")
print(f" Dimensions finales : {df_final.shape}")
print(f" Nombre de compétences uniques : {len([col for col in df_final.columns if col not in df_merged.columns])}")

 Transformation réussie !
 Dimensions finales : (5396, 141)
 Nombre de compétences uniques : 125


In [50]:
colonnes = df_final.columns
print(colonnes)

Index(['intitule_poste', 'nom_entreprise', 'date_publication', 'type_contrat',
       'niveau_seniorite', 'description_poste', 'source_offre', 'teletravail',
       'metier_recherche', 'id_offre',
       ...
       'cybersécurité', 'spring boot', 'devops', 'next.js', 'prometheus',
       'grafana', 'pytorch', 'keras', 'tensorflow', 'siem'],
      dtype='object', length=141)


In [52]:
df_final.to_csv('df_final.csv', index=False)